In [1]:
import sys, os
import numpy as np
import pandas as pd
import trimesh
import polyscope as ps
import time

In [48]:
notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, ".."))
sys.path.append(parent_dir)
# export_dir = "Z:\IRI_2026\clothilde-sim\python_code\exported_data3"

EXPORT_DIR = parent_dir + "\\exported_data3_binary"

from implementation.Cloth_speed import Cloth
from implementation.utils import createRectangularMesh
from implementation.Gripper import (
    SimulateGripper, 
    quat_transform_points,
    quat_normalize,
    quat_from_axis_angle
)

In [54]:
replay_file_path = EXPORT_DIR + "\\replay_cloth.bin"
replay_file_path

'z:\\IRI_2026\\clothilde-sim\\python_code\\exported_data3_binary\\replay_cloth.bin'

In [55]:
f = open(replay_file_path, "rb") # rb: read binary
magic = f.read(8).decode("ascii")

if magic != "CLTHSIM1":
        raise ValueError(f"Wrong file format: {magic}")

version = np.fromfile(f, dtype="<i4", count=1)[0]
# <:   little-endian i4: 4-byte integer (can store upto 4 * 8 = 32 bits)
if version != 1:
        raise ValueError(f"Unsupported version: {version}")

header = np.fromfile(f, dtype="<i4", count=4)
n_frames, n_vertices, n_faces, n_params = header
print(f'n_frames: {n_frames}, n_vertices: {n_vertices}, n_faces: {n_faces}, n_params: {n_params}')

n_frames: 866, n_vertices: 441, n_faces: 400, n_params: 14


In [56]:
mesh_vertices_bin = np.fromfile(f, dtype="<f8", count=n_vertices * 3)
mesh_vertices = mesh_vertices_bin.reshape(n_vertices, 3)

mesh_faces_bin = np.fromfile(f, dtype="<i4", count=n_faces * 4)
mesh_faces = mesh_faces_bin.reshape(n_faces, 4)

In [57]:
cloth_frames_bin = np.fromfile(f, dtype="<f8", count=n_frames * n_vertices * 3)
cloth_frames = cloth_frames_bin.reshape(n_frames, n_vertices, 3)

In [58]:
gripper_poses_bin = np.fromfile(f, dtype="<f4", count=n_frames * 7)
gripper_poses = gripper_poses_bin.reshape(n_frames, 7)

jaw_status_bin = np.fromfile(f, dtype="<i4", count=n_frames)
jaw_status = jaw_status_bin.reshape(n_frames)

In [59]:
times = np.fromfile(f, dtype="<f4", count=n_frames)

sim_params_arr = np.fromfile(f, dtype="<f4", count=n_params)

param_names = [
"dt", "tol", "sub_steps", "rho", "delta", "alpha",
"kappa", "kappa_bnd", "str", "shr", "slf",
"mu_f", "mu_s", "thck"
]
params = dict(zip(param_names, sim_params_arr))
params["sub_steps"] = int(params["sub_steps"])